# CRAC — Coral Reef Health Classifier (training)

Fine-tunes a **ResNet-18 CNN** to label a reef image as `healthy`, `bleached`, `dead`, or `algae_covered`,
then exports the trained model for the CRAC dashboard to serve.

**Runtime → Change runtime type → T4 GPU** before you start. On a T4 a full run is a few minutes.

What this notebook does:
1. Clone the repo and install dependencies
2. Get a labelled image set (`demo`, your Google Drive, or Kaggle)
3. Build the manifest and split it **by site** — no site appears in two splits
4. Baseline first, then train the CNN
5. Evaluate honestly: per-class precision/recall/F1 and the confusion matrix
6. Export `data/models/<version>/` for the API to serve

## 1 · Setup

In [ ]:
REPO_URL = "https://github.com/Nathan-Dela-Pena/Coral-Reef-Assessment-Classifier-CRAC-.git"
BRANCH   = "main"

import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = Path("/content/CRAC") if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{REPO_DIR}[model,dev]"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("working in", Path.cwd())

In [ ]:
import torch
from crac.model.dataset import LABELS
from crac.model.train import pick_device

DEVICE = pick_device()
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then rerun this cell.")
print("labels:", LABELS)

## 2 · Get a labelled image set

Pick one `DATA_SOURCE`:

| value | what it does |
|---|---|
| `"demo"` | generates synthetic images locally. Proves the pipeline end to end in ~1 min. **The resulting accuracy means nothing about real reefs.** |
| `"drive"` | reads a folder in your Google Drive |
| `"kaggle"` | downloads a Kaggle dataset (needs `kaggle.json`) |

Whichever you pick, the folder must have **one subdirectory per label**:

```
images/
  healthy/  bleached/  dead/  algae_covered/
```

Name files `site=<site_id>_<YYYY-MM-DD>_0001.jpg` where you can. The site is what
keeps the train/test split honest — frames from one dive are near-duplicates, and
a random split leaks them across the boundary and inflates test accuracy badly.

In [ ]:
DATA_SOURCE = "demo"           # "demo" | "drive" | "kaggle"

DRIVE_IMAGE_DIR = "/content/drive/MyDrive/CRAC/images"
KAGGLE_DATASET  = "sohaibanwaar1203/coral-reef-dataset"   # any dataset with per-label folders
DEMO_PER_CLASS  = 150

from pathlib import Path
import subprocess, sys

if DATA_SOURCE == "demo":
    IMAGE_DIR = Path("data/raw/images")
    subprocess.run([sys.executable, "scripts/make_demo_images.py",
                    "--per-class", str(DEMO_PER_CLASS), "--out", str(IMAGE_DIR)], check=True)

elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    IMAGE_DIR = Path(DRIVE_IMAGE_DIR)
    assert IMAGE_DIR.is_dir(), f"{IMAGE_DIR} not found in your Drive"

elif DATA_SOURCE == "kaggle":
    # Upload kaggle.json (Kaggle -> Settings -> Create New Token) when prompted.
    from google.colab import files
    import os, json
    if not Path("/root/.kaggle/kaggle.json").exists():
        files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        os.replace("kaggle.json", "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
                    "-p", "data/raw", "--unzip"], check=True)
    IMAGE_DIR = Path("data/raw")          # adjust to the folder that holds the label dirs
    print("Downloaded. Inspect the tree and point IMAGE_DIR at the label folders:")
    for p in sorted(IMAGE_DIR.rglob("*"))[:40]:
        if p.is_dir():
            print("  ", p)
else:
    raise ValueError(DATA_SOURCE)

print("IMAGE_DIR =", IMAGE_DIR)

### Build the manifest

`build_manifest.py` crawls the folders once and writes `(image_path, label, site_id, captured_on)`.
Folder names are matched against the taxonomy, with a few common aliases (`algae` → `algae_covered`,
`rubble` → `dead`). Anything unrecognised is skipped loudly rather than silently mislabelled.

In [ ]:
MANIFEST = Path("data/processed/manifest.csv")
subprocess.run([sys.executable, "scripts/build_manifest.py", str(IMAGE_DIR),
                "-o", str(MANIFEST)], check=True)

import pandas as pd
manifest = pd.read_csv(MANIFEST)
display(manifest["label"].value_counts().rename("images"))
print("sites:", sorted(manifest["site_id"].dropna().astype(str).unique())[:12])

In [ ]:
# Look at the data before you train on it. Mislabelled folders are the single
# most common reason a run looks fine and means nothing.
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(len(LABELS), 5, figsize=(11, 2.4 * len(LABELS)))
for row, label in enumerate(LABELS):
    sample = manifest[manifest.label == label].head(5)
    for col in range(5):
        ax = axes[row][col]
        ax.axis("off")
        if col < len(sample):
            ax.imshow(Image.open(sample.iloc[col]["image_path"]).convert("RGB"))
        if col == 0:
            ax.set_title(label, loc="left", fontsize=11)
fig.tight_layout()
plt.show()

## 3 · Split by site

Grouped split, not a random one. `split_by_site` shuffles **groups** (survey, dive,
or site — whichever the manifest records) and assigns whole groups to train/val/test.
If the manifest has fewer than three groups it falls back to a row split and warns you;
treat any accuracy from that fallback as optimistic.

In [ ]:
from crac.model.dataset import split_by_site

train_df, val_df, test_df = split_by_site(MANIFEST, val_frac=0.15, test_frac=0.15, seed=0)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    sites = sorted(d["site_id"].dropna().astype(str).unique())
    print(f"{name:>5}: {len(d):5d} images | sites: {sites}")

overlap = set(train_df.site_id) & set(test_df.site_id)
print("\ntrain/test site overlap:", overlap or "none (good)")

## 4 · Baseline first

> "The CNN got 82%" is meaningless on its own.

With four roughly balanced classes, always guessing the majority class scores ~25%.
On a realistic reef dataset — where most frames are healthy — it can score 70%+, and
that is the number your CNN actually has to beat.

In [ ]:
from crac.model.train import majority_class_baseline

baseline = majority_class_baseline(train_df, test_df)
print(f"majority-class baseline: predicts '{baseline['predicted_label']}' "
      f"every time -> accuracy {baseline['accuracy']:.3f}")

## 5 · Train

Two stages, which is what makes transfer learning work on a small dataset:

1. **Frozen backbone** (`head_epochs`) — only the new 4-way classification head learns.
   The pretrained features stay intact while the head is still producing noise.
2. **Fine-tune** — unfreeze everything at a 10× lower learning rate.

The loss is inverse-frequency weighted, because class imbalance here is guaranteed and
plain accuracy would happily ignore `bleached` — the one class the project exists to detect.

In [ ]:
EPOCHS       = 12
HEAD_EPOCHS  = 3
LR           = 1e-3
FINE_TUNE_LR = 1e-4
BATCH_SIZE   = 32
VERSION      = None     # None -> resnet18-<timestamp>

from crac.model.train import train

version, metrics = train(
    MANIFEST,
    epochs=EPOCHS,
    head_epochs=HEAD_EPOCHS,
    lr=LR,
    fine_tune_lr=FINE_TUNE_LR,
    batch_size=BATCH_SIZE,
    out_version=VERSION,
    workers=2,
)
print("\nmodel version:", version)

In [ ]:
# Training curves. Val loss turning up while train loss keeps falling = overfitting;
# cut EPOCHS back to where val loss bottomed out, or add data.
import matplotlib.pyplot as plt

hist = pd.DataFrame(metrics["history"])
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.6))
a.plot(hist.epoch, hist.train_loss, label="train"); a.plot(hist.epoch, hist.val_loss, label="val")
a.set_title("loss"); a.set_xlabel("epoch"); a.legend()
b.plot(hist.epoch, hist.train_acc, label="train"); b.plot(hist.epoch, hist.val_acc, label="val")
b.axhline(baseline["accuracy"], ls="--", c="grey", label="majority baseline")
b.set_title("accuracy"); b.set_xlabel("epoch"); b.legend()
if HEAD_EPOCHS < EPOCHS:
    for ax in (a, b):
        ax.axvline(HEAD_EPOCHS + 0.5, c="crimson", lw=1, alpha=0.6)
fig.tight_layout(); plt.show()

## 6 · Evaluate honestly

Accuracy alone hides the failure that matters: a model that scores well overall while
recall on `bleached` sits near zero. Read the per-class rows, not the headline.

In [ ]:
rows = [{"label": k, **{m: v[m] for m in ("precision", "recall", "f1-score", "support")}}
        for k, v in metrics["per_class"].items()]
display(pd.DataFrame(rows).set_index("label").round(3))

print(f"accuracy  {metrics['accuracy']:.3f}   (baseline {baseline['accuracy']:.3f})")
print(f"macro-F1  {metrics['macro_f1']:.3f}")
print("test sites:", metrics["test_sites"])

cm = pd.DataFrame(metrics["confusion_matrix"]["counts"],
                  index=[f"true {l}" for l in LABELS],
                  columns=[f"pred {l}" for l in LABELS])
display(cm)

In [ ]:
from IPython.display import Image as IPyImage
from crac import config

cm_png = Path(config.MODELS_DIR) / version / "confusion_matrix.png"
display(IPyImage(filename=str(cm_png))) if cm_png.exists() else print("no confusion_matrix.png")

## 7 · Try it on one image

The same `predict` path the API's `/classify` endpoint uses, so what you see here is
exactly what the dashboard will show.

In [ ]:
from crac.model.predict import load, predict_image

predictor = load(version)
sample = test_df.sample(4, random_state=0) if len(test_df) >= 4 else test_df

fig, axes = plt.subplots(1, len(sample), figsize=(3.1 * len(sample), 3.6))
axes = [axes] if len(sample) == 1 else axes
for ax, (_, row) in zip(axes, sample.iterrows()):
    result = predict_image(predictor, row["image_path"])
    ax.imshow(Image.open(row["image_path"]).convert("RGB")); ax.axis("off")
    ok = "OK" if result["label"] == row["label"] else "MISS"
    ax.set_title(f"{ok}  pred {result['label']} {result['confidence']:.0%}\ntrue {row['label']}",
                 fontsize=9)
fig.tight_layout(); plt.show()

In [ ]:
# Upload your own reef photo and classify it (Colab only).
if IN_COLAB:
    from google.colab import files
    from crac.model.predict import predict_pil
    uploaded = files.upload()
    for name, blob in uploaded.items():
        import io
        img = Image.open(io.BytesIO(blob)).convert("RGB")
        r = predict_pil(predictor, img)
        plt.figure(figsize=(4, 4)); plt.imshow(img); plt.axis("off")
        plt.title(f"{r['label']} ({r['confidence']:.0%}) — "
                  f"{'HEALTHY' if r['healthy'] else 'UNHEALTHY'}")
        plt.show()
        print({k: round(v, 3) for k, v in r["probabilities"].items()})

## 8 · Export the trained model

The dashboard serves whatever sits in `data/models/<version>/`. Download the zip, unzip it
into that folder in your local checkout, restart the API, and the website picks it up.

In [ ]:
import shutil
from crac import config

model_dir = Path(config.MODELS_DIR) / version
print("contents of", model_dir)
for f in sorted(model_dir.iterdir()):
    print(f"  {f.name:<24} {f.stat().st_size/1024:8.1f} KB")

archive = shutil.make_archive(f"/content/{version}" if IN_COLAB else f"/tmp/{version}",
                              "zip", root_dir=model_dir.parent, base_dir=version)
print("\nwrote", archive)

if IN_COLAB:
    from google.colab import files
    files.download(archive)

In [ ]:
# Optional: also save it to Google Drive so it survives the runtime being recycled.
SAVE_TO_DRIVE = False

if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/CRAC/models") / version
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(model_dir, dest, dirs_exist_ok=True)
    print("copied to", dest)

---

### Next steps

```bash
unzip <version>.zip -d data/models/     # into your local checkout
python scripts/serve.py                 # http://127.0.0.1:8000
```

**Before you report any number from this notebook**: if you trained on `demo` data, the
accuracy describes synthetic textures, not coral. Retrain on a real labelled source
(CoralNet, the Moorea LTER MLC set, or NOAA NCRMP imagery) and re-read the per-class
recall on `bleached` — that row is the one the whole project stands on.